In [ ]:
%matplotlib inline
import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['font.family'] = 'monospace'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['figure.facecolor'] = '#fafafa'
mpl.rcParams['axes.facecolor'] = '#fafafa'

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(2021)

n_weeks = 156
dates = pd.date_range(start='2021-01-04', periods=n_weeks, freq='W-MON')

# --- IBEX 35 price series (random walk, range 8200-9800) ---
price_start = 9000
steps = np.random.normal(0, 50, n_weeks)
prices_raw = price_start + np.cumsum(steps)
# rescale to [8200, 9800]
prices_min = prices_raw.min()
prices_max = prices_raw.max()
ibex_prices = 8200 + (prices_raw - prices_min) / (prices_max - prices_min) * 1600

# --- Weekly returns ---
ibex_returns = np.diff(ibex_prices) / ibex_prices[:-1]
ibex_returns = np.append(ibex_returns, np.nan)

# --- Sentiment scores (-1 to 1) ---
# lag-1 correlation injected by construction: sentiment at t encodes next week's return direction
noise = np.random.normal(0, 0.3, n_weeks)
signal = np.roll(ibex_returns, 1)  # sentiment[t] correlates with return[t+1]
signal[0] = 0
signal[-1] = 0
sentiment_raw = 0.6 * signal / (np.nanstd(signal) + 1e-9) + noise
# clip to [-1, 1]
sentiment = np.clip(sentiment_raw, -1, 1)

df = pd.DataFrame({
    'date': dates,
    'ibex_price': ibex_prices,
    'ibex_return': ibex_returns,
    'sentiment': sentiment
})

print('Shape:', df.shape)
print(df.head())